In [1]:
# autoload
%load_ext autoreload
%autoreload 2

In [2]:
basedir = "/Users/rikhoekstra/develop/republic_ner_matching"
data_dir = f"{basedir}/data"
output_dir = f"{basedir}/output"



In [6]:
import os
import pandas as pd
from pathlib import Path

data_path = Path(data_dir)
df_res = pd.read_json(data_path / "enriched_resolutions_1626_1630_complete.json")
df_res_flat = pd.read_parquet(data_path / "resolutions_flat.parquet")
#entities
df_loc = pd.read_json(data_path / "LOC-entities.json")
df_per = pd.read_json(data_path / "PER-entities.json")
df_org = pd.read_json(data_path / "ORG-entities.json") 

In [11]:
all_places = df_res['places'].explode()
all_places.value_counts()


places
Holland          1121
Frankrijk         559
Engeland          528
Amsterdam         504
Hertogenbosch     454
                 ... 
Bretagne            1
Blanet              1
Oost-Vlieland       1
Blauwe Sluis        1
Maasbommel          1
Name: count, Length: 1507, dtype: int64

In [12]:
all_people = df_res['persons'].explode()
all_people.value_counts()


persons
457922    674
90194     604
168184    533
580636    531
789512    492
         ... 
379051      1
295835      1
114036      1
658106      1
436378      1
Name: count, Length: 6638, dtype: int64

In [27]:

persons = all_people[all_people.notna()]
persons = persons.to_frame()
persons.columns = ['person_id']
persons.index.name = 'resolution_id'
persons.person_id = persons.person_id.astype(int)
persons.head(10)

,person_id
resolution_id,
1,791967
3,168184
3,124646
3,125152
4,494421
4,930606
4,727188
5,920048
5,729000


In [28]:
all_institutions = df_res['institutions'].explode()
all_institutions.value_counts()

institutions
33        1126
34         935
36         562
29         548
78         399
          ... 
369588       1
91           1
419009       1
789512       1
63           1
Name: count, Length: 95, dtype: int64

In [29]:
persons_info.dtypes

Id_persoon    int64
fullname        str
short_name      str
dtype: object

In [33]:
#we have to resolve places and institutions
persons_info = pd.read_json(data_path / "persons_info.json")
person_w_name = persons.merge(persons_info, how='left', left_on='person_id', right_on='Id_persoon')
person_w_name.drop(columns=['Id_persoon'], inplace=True)
print(len(person_w_name))
person_w_name.head(10)


38153


,person_id,fullname,short_name
0,791967,"Huygen, Rutger heer van Clarenbeek","Huygen, Rutger heer van Clarenbeek"
1,168184,"Schaffer, Goossen","Schaffer, Goossen"
2,124646,"Carleton, Dudley","Carleton, Dudley"
3,125152,"Vane, Henry","Vane, Henry"
4,494421,"van Randwijck tot Bemmel, Arnold","van Randwijck tot Bemmel, Arnold"
5,930606,"de Bar, Nicolas heer van Baugy","de Bar, Nicolas heer van Baugy"
6,727188,"Pissot, Jan","Pissot, Jan"
7,920048,"Boormaecker, Govert Govertsz.","Boormaecker, Govert Govertsz."
8,729000,"Roos, Gerrit Evertsz.","Roos, Gerrit Evertsz."
9,614058,"van Nieucoop, Gerrit Willemsz.","van Nieucoop, Gerrit Willemsz."


In [46]:
# now the same for institutions
import json
import pandas as pd

with open("/Users/rikhoekstra/develop/republic_ner_matching/data/instelling_info.json", "r", encoding="utf-8") as f:
    data = json.load(f)

institutions_info = pd.DataFrame(data["instelling"])
institutions_info.ID_instelling = institutions_info.ID_instelling.astype("str", errors='ignore')
institutions = all_institutions[all_institutions.notna()]
institutions = institutions.to_frame()
institutions.columns = ['institution_id']
# institutions.institution_id = institutions.institution_id.astype("Int64", errors='ignore')
institutions.index.name = 'resolution_id'
# institutions_info
institution_w_name = institutions.merge(institutions_info, how='left', left_on='institution_id', right_on='ID_instelling')
institution_w_name.drop(columns=['ID_instelling'], inplace=True)
print(len(institution_w_name))
institution_w_name.head(10)

7457


,institution_id,naam,primair,verwijzing,repertorium,DateAdded,DateChanged,personadded,personchanged
0,14,Hof van Holland en Zeeland,1.0,0.0,0.0,2002-07-31,2002-07-31,Nobody,Nobody
1,34,Admiraliteit te Amsterdam,1.0,0.0,0.0,2002-07-31,2002-07-31,Nobody,Nobody
2,13,Hoge Raad van Holland en Zeeland,1.0,0.0,0.0,2002-07-31,2002-07-31,Nobody,Nobody
3,14,Hof van Holland en Zeeland,1.0,0.0,0.0,2002-07-31,2002-07-31,Nobody,Nobody
4,78,WIC (Heren Negentien/Bewindhebbers van de WIC),1.0,0.0,0.0,2002-07-31,2002-11-14,Nobody,Ida
5,35,Admiraliteit in Zeeland,1.0,0.0,0.0,2002-07-31,2002-07-31,Nobody,Nobody
6,52,Gewestelijke Staten van Zeeland,1.0,0.0,0.0,2002-07-31,2002-07-31,Nobody,Nobody
7,33,Admiraliteit op de Maze,1.0,0.0,0.0,2002-07-31,2002-07-31,Nobody,Nobody
8,34,Admiraliteit te Amsterdam,1.0,0.0,0.0,2002-07-31,2002-07-31,Nobody,Nobody
9,51,Gewestelijke Staten van Gelderland,1.0,0.0,0.0,2002-07-31,2002-07-31,Nobody,Nobody


In [52]:
# ok now we match these to the entities in the df_loc, df_per, df_org dataframes. We can do this by matching the names.


inst_matched = df_org.merge(institution_w_name, how='left', left_on='name', right_on='naam')
print(len(inst_matched))
display(inst_matched.head(10))


1059


,id,name,category,labels,comment,links,institution_id,naam,primair,verwijzing,repertorium,DateAdded,DateChanged,personadded,personchanged
0,O0000002,Admiraliteit te Londen,ORG,"[Europa, Admiraliteit, Zeevaart, Landelijk]",NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,O0000003,Admiraliteit van Amsterdam,ORG,"[Regionaal, Republiek, Oorlog, Admiraliteit, Z...",NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,O0000004,Admiraliteit van Friesland,ORG,"[Regionaal, Republiek, Oorlog, Admiraliteit, Z...",NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,O0000005,Admiraliteit van Rotterdam,ORG,"[Regionaal, Republiek, Oorlog, Admiraliteit, Z...",NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,O0000006,Admiraliteit van Westfriesland,ORG,"[Regionaal, Republiek, Oorlog, Admiraliteit, Z...",NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,O0000007,Admiraliteit van Zeeland,ORG,"[Regionaal, Republiek, Oorlog, Admiraliteit, Z...",NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,O0000008,Admiraliteiten in Denemarken,ORG,"[Europa, Admiraliteit, Zeevaart, Landelijk]",NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,O0000009,Admiraliteiten in Frankrijk,ORG,"[Europa, Admiraliteit, Zeevaart, Landelijk]",NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,O0000010,Admiraliteiten in Pruissen,ORG,"[Europa, Admiraliteit, Zeevaart, Landelijk]",NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,O0000011,Admiraliteiten in Zweden,ORG,"[Europa, Admiraliteit, Zeevaart, Landelijk]",NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [57]:
rvs_id = inst_matched[inst_matched['naam'].str.contains("Raad van State", na=False)].institution_id.iat[0]
resolutions_with_rvs = inst_matched[inst_matched['institution_id'] == rvs_id]
resolutions_with_rvs

,id,name,category,labels,comment,links,institution_id,naam,primair,verwijzing,repertorium,DateAdded,DateChanged,personadded,personchanged
527,O0000167,Raad van State,ORG,"[Republiek, Bestuur, Landelijk, Generaliteit]",NaN,[],31,Raad van State,1.0,0.0,0.0,2002-07-31,2002-07-31,Nobody,Nobody
528,O0000167,Raad van State,ORG,"[Republiek, Bestuur, Landelijk, Generaliteit]",NaN,[],31,Raad van State,1.0,0.0,0.0,2002-07-31,2002-07-31,Nobody,Nobody
529,O0000167,Raad van State,ORG,"[Republiek, Bestuur, Landelijk, Generaliteit]",NaN,[],31,Raad van State,1.0,0.0,0.0,2002-07-31,2002-07-31,Nobody,Nobody
530,O0000167,Raad van State,ORG,"[Republiek, Bestuur, Landelijk, Generaliteit]",NaN,[],31,Raad van State,1.0,0.0,0.0,2002-07-31,2002-07-31,Nobody,Nobody
531,O0000167,Raad van State,ORG,"[Republiek, Bestuur, Landelijk, Generaliteit]",NaN,[],31,Raad van State,1.0,0.0,0.0,2002-07-31,2002-07-31,Nobody,Nobody
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
820,O0000167,Raad van State,ORG,"[Republiek, Bestuur, Landelijk, Generaliteit]",NaN,[],31,Raad van State,1.0,0.0,0.0,2002-07-31,2002-07-31,Nobody,Nobody
821,O0000167,Raad van State,ORG,"[Republiek, Bestuur, Landelijk, Generaliteit]",NaN,[],31,Raad van State,1.0,0.0,0.0,2002-07-31,2002-07-31,Nobody,Nobody
822,O0000167,Raad van State,ORG,"[Republiek, Bestuur, Landelijk, Generaliteit]",NaN,[],31,Raad van State,1.0,0.0,0.0,2002-07-31,2002-07-31,Nobody,Nobody
823,O0000167,Raad van State,ORG,"[Republiek, Bestuur, Landelijk, Generaliteit]",NaN,[],31,Raad van State,1.0,0.0,0.0,2002-07-31,2002-07-31,Nobody,Nobody


In [58]:
org_annotations = pd.read_json(data_path / "ORG-annotations.json")

In [78]:
import json
import pandas as pd

# load ORG annotations
with open("/Users/rikhoekstra/develop/republic_ner_matching/data/ORG-annotations.json", "r", encoding="utf-8") as f:
    org_data = json.load(f)

org_df = pd.DataFrame(
    {
        "entity_nr": [item["provenance"]["target"][-1] for item in org_data],
        "inv": [int(item["reference"]["inv"]) for item in org_data],
        "tag_text": [item["reference"]["tag_text"] for item in org_data],
        "resolution_id": [item["reference"]["resolution_id"] for item in org_data],
        "paragraph_id": [item["reference"]["paragraph_id"] for item in org_data],
        "offset": [item["reference"]["offset"] for item in org_data],
        "end": [item["reference"]["end"] for item in org_data],
    }
)

# load inventory metadata
with open("/Users/rikhoekstra/develop/republic_ner_matching/data/inventory_metadata.json", "r", encoding="utf-8") as f:
    inventory_meta = json.load(f)

meta_df = (
    pd.DataFrame(inventory_meta)[["inventory_num", "year"]]
    .explode("year")
    .dropna(subset=["year"])
    .drop_duplicates()
)

# merge year onto annotations
org_df = org_df.merge(meta_df, left_on="inv", right_on="inventory_num", how="left").drop(columns=["inventory_num"])
org_df['entity_nr'] = org_df['entity_nr'].str.extract(r'(O\d+)')

In [85]:
rvs_oid = inst_matched[inst_matched['naam'].str.contains("Raad van State", na=False)].id.iat[0]
rvs_resolutions_raw = org_df[org_df['entity_nr'] == rvs_oid]

In [88]:
df_res_flat.head(5)

,id,type,date,year,weekday,paragraph_texts,resolutions_text
0,session-3788-num-1-resolution-1,resolution,1733-01-02,1733,vrijdag,"[""ONtfangen een Missive van den Resident Spina...","ONtfangen een Missive van den Resident Spina, ..."
1,session-3788-num-1-resolution-10,resolution,1733-01-02,1733,vrijdag,"[""IS ter Vergaderinge gelesen de Requeste van ...",IS ter Vergaderinge gelesen de Requeste van de...
2,session-3788-num-1-resolution-11,resolution,1733-01-02,1733,vrijdag,"[""17 Ynde ter Vergaderinge getoont en geexhibe...",17 Ynde ter Vergaderinge getoont en geexhibeer...
3,session-3788-num-1-resolution-12,resolution,1733-01-02,1733,vrijdag,"[""17 Ynde ter Vergaderinge getoont ende geëxhi...",17 Ynde ter Vergaderinge getoont ende geëxhibe...
4,session-3788-num-1-resolution-13,resolution,1733-01-02,1733,vrijdag,"[""OP de Requeste van de gesamentlijcke Straatm...",OP de Requeste van de gesamentlijcke Straatmaa...


In [90]:
df_res_flat[(df_res_flat['id'].isin(rvs_resolutions_raw['resolution_id'])) & (df_res_flat['year'].between(1626,1630))]

,id,type,date,year,weekday,paragraph_texts,resolutions_text
84104,session-3186-num-10-resolution-1,resolution,1627-01-13,1627,woensdag,"[""Opt versoeck van Claes Cornelissen Meester t...",Opt versoeck van Claes Cornelissen Meester tim...
84106,session-3186-num-10-resolution-11,resolution,1627-01-13,1627,woensdag,"[""Synde byden Raedt van State geadviseert dat ...",Synde byden Raedt van State geadviseert dat ae...
84107,session-3186-num-10-resolution-12,resolution,1627-01-13,1627,woensdag,"[""Het versoeck van Capitein Pitcarne om te heb...",Het versoeck van Capitein Pitcarne om te hebbe...
84115,session-3186-num-10-resolution-2,resolution,1627-01-13,1627,woensdag,"[""Synde byden Raedt van State geadviseert, dat...","Synde byden Raedt van State geadviseert, dat h..."
84116,session-3186-num-10-resolution-20,resolution,1627-01-13,1627,woensdag,"[""Is goetgevonden dat facti specis sal werden ...",Is goetgevonden dat facti specis sal werden ov...
...,...,...,...,...,...,...,...
577343,session-3189-num-97-resolution-8,resolution,1630-04-17,1630,woensdag,"[""Ontfangen noch een missive vande selve gesch...",Ontfangen noch een missive vande selve geschre...
577344,session-3189-num-97-resolution-9,resolution,1630-04-17,1630,woensdag,"[""Het versoeck van Capn. de Loges om te mogen ...",Het versoeck van Capn. de Loges om te mogen ex...
577351,session-3189-num-98-resolution-6,resolution,1630-04-18,1630,donderdag,"[""Ontfangen een missive vande heeren Burgermrs...",Ontfangen een missive vande heeren Burgermrs e...
577360,session-3189-num-99-resolution-14,resolution,1630-04-19,1630,vrijdag,"[""Synde gelesen de requeste van Jacob van Nisp...",Synde gelesen de requeste van Jacob van Nispe ...


In [ ]:
# Step 1: add flat resolution dates to every org annotation
flat_dates = df_res_flat[['id', 'date']].rename(columns={'id': 'resolution_id', 'date': 'flat_date'})
org_df_dated = org_df.merge(flat_dates, on='resolution_id', how='left')
org_df_dated['flat_date'] = pd.PeriodIndex(org_df_dated['flat_date'].astype(str), freq='D')

print(f"org_df_dated: {len(org_df_dated)} rows, {org_df_dated['flat_date'].isna().sum()} without date")
org_df_dated.head(5)


org_df_dated: 507268 rows, 0 without date


,entity_nr,inv,tag_text,resolution_id,paragraph_id,offset,end,year,flat_date
0,O0011985,3097,chambre des arydes,session-3097-num-10-resolution-1,session-3097-num-10-para-3,33,51,1577,1577-05-29
1,O0012188,3097,Conseil de Brabant,session-3097-num-100-resolution-5,session-3097-num-100-para-8,213,231,1577,1577-08-26
2,O0012188,3097,Conseil de Brabant,session-3097-num-117-resolution-2,session-3097-num-117-para-4,80,98,1577,1577-09-12
3,O0012188,3097,Conseil de Brabant,session-3097-num-163-resolution-7,session-3097-num-163-para-10,184,202,1577,1577-10-30
4,O0012188,3099,Conseil de Brabant,session-3099-num-110-resolution-6,session-3099-num-110-para-8,76,94,1577,1578-03-23


In [ ]:
# Step 2: build date-based resolution refs and anchor candidates for 1626-1630
anchor_window_start = 1626
anchor_window_end = 1630

flat_window = (
    df_res_flat.loc[
        df_res_flat['year'].between(anchor_window_start, anchor_window_end),
        ['id', 'date', 'year'],
    ]
    .copy()
    .sort_values(['date', 'id'])
)
flat_window['date_period'] = pd.PeriodIndex(flat_window['date'].astype(str), freq='D')
flat_window['sequence_nr'] = flat_window.groupby('date_period').cumcount() + 1
flat_window['resolution_ref'] = (
    flat_window['date_period'].astype(str)
    + '-'
    + flat_window['sequence_nr'].astype(str).str.zfill(2)
)

org_window = org_df_dated.merge(
    flat_window[['id', 'resolution_ref']],
    left_on='resolution_id',
    right_on='id',
    how='inner',
)
org_window = org_window[org_window['year'].between(anchor_window_start, anchor_window_end)].copy()
org_window['anchor_ref'] = org_window['resolution_ref']
org_window['anchor_label'] = org_window['tag_text']

print(f"Anchor candidates in {anchor_window_start}-{anchor_window_end}: {len(org_window)}")
print(f"Unique resolution refs: {org_window['anchor_ref'].nunique()}")
print(f"Unique orgs: {org_window['entity_nr'].nunique()}")
org_window[['resolution_id', 'anchor_ref', 'entity_nr', 'anchor_label', 'year', 'flat_date', 'offset', 'end']].head(10)


Anchor candidates in 1626-1630: 9678
Unique resolution refs: 6294
Unique orgs: 76


,resolution_id,anchor_ref,entity_nr,anchor_label,year,flat_date,offset,end
0,session-3186-num-115-resolution-4,1627-07-04-34,O0012259,Estats Generaulx,1627,1627-07-04,14,30
1,session-3187-num-45-resolution-1,1628-02-17-01,O0012259,Estats Generaulx,1628,1628-02-17,152,168
3,session-3185-num-28-resolution-23,1626-02-12-16,O0012259,Messieurs les Estats Generaulx,1626,1626-02-12,266,296
4,session-3186-num-110-resolution-6,1627-06-22-32,O0012259,Messieurs les Estats Generaulxle,1627,1627-06-22,71,103
5,session-3189-num-282-resolution-18,1630-11-13-10,O0012259,Messieurs les Estatz generaulx,1630,1630-11-13,393,423
6,session-3186-num-145-resolution-7,1627-08-26-29,O0012247,Staten van Brabant,1627,1627-08-26,277,295
7,session-3186-num-21-resolution-7,1627-02-04-15,O0012247,Staten van Brabant,1627,1627-02-04,398,416
8,session-3186-num-34-resolution-2,1627-02-22-02,O0012247,Staten van Brabant,1627,1627-02-22,273,291
9,session-3187-num-147-resolution-18,1628-06-14-10,O0012247,Staten van Brabant,1628,1628-06-14,341,359
10,session-3187-num-164-resolution-8,1628-07-05-23,O0012247,Staten van Brabant,1628,1628-07-05,437,455


In [ ]:
def unique_texts(series):
    return sorted({str(value) for value in series if pd.notna(value) and str(value).strip() and str(value).strip().lower() != 'nan'})

resolution_window = df_res[['date']].reset_index().rename(columns={'index': 'resolution_id'}).copy()
resolution_window['date'] = pd.PeriodIndex(resolution_window['date'].astype(str), freq='D')
resolution_window = resolution_window[resolution_window['date'].dt.year.between(anchor_window_start, anchor_window_end)].copy()
resolution_window = resolution_window.sort_values(['date', 'resolution_id'])
resolution_window['date_period'] = resolution_window['date']
resolution_window['sequence_nr'] = resolution_window.groupby('date_period').cumcount() + 1
resolution_window['resolution_ref'] = (
    resolution_window['date_period'].astype(str)
    + '-'
    + resolution_window['sequence_nr'].astype(str).str.zfill(2)
)

place_annotations = all_places[all_places.notna()].to_frame().reset_index()
place_annotations.columns = ['resolution_id', 'place_id']
place_annotations['resolution_id'] = place_annotations['resolution_id'].astype(int)
place_annotations['place_id'] = place_annotations['place_id'].astype(str)

place_window = place_annotations.merge(
    df_loc[['id', 'name']],
    how='left',
    left_on='place_id',
    right_on='id',
)
place_window.drop(columns=['id'], inplace=True)
place_window['anchor_label'] = place_window['name'].fillna(place_window['place_id'])
place_window = place_window.merge(
    resolution_window[['resolution_id', 'resolution_ref', 'date_period']],
    on='resolution_id',
    how='inner',
)
place_window['anchor_ref'] = place_window['resolution_ref']

org_annotations = inst_matched.copy()
if 'resolution_id' not in org_annotations.columns:
    org_annotations = org_annotations.reset_index()
    if 'index' in org_annotations.columns and 'resolution_id' not in org_annotations.columns:
        org_annotations = org_annotations.rename(columns={'index': 'resolution_id'})
org_annotations['resolution_id'] = org_annotations['resolution_id'].astype(int)
org_annotations = org_annotations[org_annotations['naam'].notna()].copy()

org_window = org_annotations.merge(
    resolution_window[['resolution_id', 'resolution_ref', 'date_period']],
    on='resolution_id',
    how='inner',
)
org_window['anchor_ref'] = org_window['resolution_ref']
org_window['anchor_label'] = org_window['naam']

place_grouped = place_window.groupby(['resolution_id', 'anchor_ref', 'date_period'], as_index=False).agg(
    place_tags=('anchor_label', unique_texts),
    place_count=('place_id', 'nunique'),
)

org_grouped = org_window.groupby(['resolution_id', 'anchor_ref', 'date_period'], as_index=False).agg(
    org_tags=('anchor_label', unique_texts),
    org_count=('institution_id', 'nunique'),
)

combined_anchor_candidates = place_grouped.merge(
    org_grouped,
    on=['resolution_id', 'anchor_ref', 'date_period'],
    how='inner',
)
combined_anchor_candidates['combined_anchor_score'] = [
    (int(place_count) * 10) + (int(org_count) * 10) + len(place_tags) + len(org_tags)
    for place_count, org_count, place_tags, org_tags in zip(
        combined_anchor_candidates['place_count'],
        combined_anchor_candidates['org_count'],
        combined_anchor_candidates['place_tags'],
        combined_anchor_candidates['org_tags'],
    )
]

print(f"Combined place/org anchor candidates in {anchor_window_start}-{anchor_window_end}: {len(combined_anchor_candidates)}")
print(f"Unique resolution refs: {combined_anchor_candidates['anchor_ref'].nunique()}")
print(f"Unique resolutions: {combined_anchor_candidates['resolution_id'].nunique()}")
combined_anchor_candidates.sort_values(
    ['combined_anchor_score', 'place_count', 'org_count', 'resolution_id'],
    ascending=[False, False, False, True],
)[['resolution_id', 'anchor_ref', 'place_count', 'org_count', 'place_tags', 'org_tags']].head(10)


Combined place/org anchor candidates in 1626-1630: 385
Unique resolution refs: 385
Unique resolutions: 385


,resolution_id,anchor_ref,place_count,org_count,place_tags,org_tags
113,276,1630-04-13-10,12,1,"[Berg, Emmerik, Friesland, Groningen, Kleef, M...",[Generaliteitsrekenkamer]
61,176,1630-04-06-20,9,1,"[Breda, Duisburg, Gravenhage, Hertogenbosch, K...",[Generaliteitsrekenkamer]
52,161,1630-04-06-05,8,1,"[Berg, Büderich, Duisburg, Essen, Lippe, Mark,...",[Generaliteitsrekenkamer]
139,312,1630-04-28-04,7,1,"[Friesland, Gelderland, Groningen, Holland, Ov...",[Generaliteitsrekenkamer]
141,314,1630-04-28-06,7,1,"[Hertogenbosch, Holland, Olinda, Pernambuco, V...",[Generaliteitsrekenkamer]
90,232,1630-04-20-15,6,1,"[Amsterdam, Holland, Oostzee, Portugal, Spanje...",[Generaliteitsrekenkamer]
275,665,1630-03-28-01,6,1,"[Breda, Demer, Hertogenbosch, Heusden, Meierij...",[Raad van State]
306,730,1630-03-16-13,6,1,"[Bergen op Zoom, Halsteren, Oude Land, Sint\n\...",[Raad van State]
262,640,1630-03-26-14,5,1,"[Alicante, Cadiz, Cartagena, Málaga, Sint Lucas]",[Raad van State]
267,648,1630-03-19-02,5,1,"[Delfzijl, Eems, Groningen, Lauwers, Ommelanden]",[Raad van State]


In [119]:
from pathlib import Path
import sys
from tqdm import tqdm

sys.path.append(str(Path(basedir) / "generate_alignment"))
from build_alignment_artifacts import run

# Generate the verify_ground_truth outputs from the notebook environment.
tqdm.pandas(desc="Generating verify_ground_truth outputs")
run(preview_limit=10, stratified_size=100, verification_page_size=30)

print("Generated verify_ground_truth outputs in:", output_dir)

Loading data files...
✓ Loaded 19134 enriched resolutions
✓ Loaded 692156 flat resolutions
✓ Loaded 2459 LOC canonical names
✓ Loaded 8076 PER canonical names
✓ Loaded 341 ORG canonical names
✓ Anchor map: 1579 confirmed date anchors
✓ Found 10 preview matches with place/org overlap
  Place/org enriched candidates scanned: 2000
  Place-only fallback candidates scanned: 2000
  Summary anchors: 6
✓ HTML report saved to: /Users/rikhoekstra/develop/republic_ner_matching/output/matched_resolutions_sample.html

📅 Stratified sampling across corpus:
   Months covered: 53
   Items per month: 1
  Progress: 50/53
✓ Stratified matches: 45
  Summary anchors: 30
✅ Ground truth exported: 45 samples
  Summary anchors: 30
  Entity counts: {'places': {'enriched': 554, 'found_in_flat': 146}, 'persons': {'enriched': 233, 'found_in_flat': 0}, 'organizations': {'enriched': 138, 'found_in_flat': 14}}
 sample_id enriched_date  flat_date  date_diff_days  is_same_day  is_summary_anchor  confidence_score  place_